### <font color='#1E90FF'>**Table of Contents** </font><a class="anchor" id='toc'></a>

- [1. Introduction](#1)
- [2. Import libraries and Set up](#2)
- [3. Load Data](#3)


<a class="anchor" id="1">

# **1. Introduction**

[Back to TOC](#toc)
</a>

**TO DO** introduzir introdução
vai ser feita uma análise ao dataset raw e depois com o dataset clean

<a class="anchor" id="2">

# **2. Import libraries and Set up**

[Back to TOC](#toc)
</a>

In [2]:
!pip install pyspark plotly "pandas>=2.2.0" "nbformat>=4.2.0"

In [3]:
# Install Java 17 (Required for Spark)
!sudo apt-get update
!sudo apt-get install -y openjdk-17-jdk-headless
!java -version

Get:1 https://cli.github.com/packages stable InRelease [3917 B]
Get:2 https://download.docker.com/linux/ubuntu noble InRelease [48.5 kB]       
Get:3 https://packages.cloud.google.com/apt cloud-sdk InRelease [1621 B]
Get:4 https://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]     
Get:5 https://download.docker.com/linux/ubuntu noble/stable amd64 Packages [65.6 kB]
Get:6 https://packages.cloud.google.com/apt cloud-sdk/main all Packages [2012 kB]
Hit:7 https://us-east-1.ec2.archive.ubuntu.com/ubuntu noble InRelease          
Get:8 https://download.docker.com/linux/ubuntu noble/stable amd64 Contents (deb) [1637 B]
Get:9 https://us-east-1.ec2.archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:10 https://archive.ubuntu.com/ubuntu noble InRelease [256 kB]              
Get:11 https://packages.cloud.google.com/apt cloud-sdk/main amd64 Packages [4735 kB]
Get:12 https://us-east-1.ec2.archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Get:13 https://archive.u

In [4]:
# Initialise the Spark Session and grab the underlying SparkContext.
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

import csv
import pyspark
from pyspark.sql import SparkSession
from pyspark.context import SparkContext

spark = SparkSession.builder \
        .master("local[*]") \
        .appName("PySpark RDDs") \
        .config("spark.executor.memory", "4g") \
        .config("spark.driver.memory", "2g") \
        .getOrCreate()

# RDDs operate through the SparkContext, not the Session.
sc = spark.sparkContext
sc.setLogLevel("ERROR")

print("Spark Session and Context ready!")
print(f"Default parallelism: {sc.defaultParallelism}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/20 16:03:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark Session and Context ready!
Default parallelism: 4


<a class="anchor" id="3">

# **3. Analysis on Raw Data**

[Back to TOC](#toc)
</a>

In [5]:
rawRDD = sc.textFile("/teamspace/studios/this_studio/Big-Data-Analytics-Project-25-26/data/FIFA/fifa21 raw data v2.csv")

print(f"Number of lines in the dataset: {rawRDD.count()}")
print(f"Default Partitions: {rawRDD.getNumPartitions()}")

Number of lines in the dataset: 93948
Default Partitions: 2


In [6]:
# Inspect the first 2 lines to see what we are dealing with.
# .take(N) brings N elements back to the driver — safer than .collect() on large RDDs.
for line in rawRDD.take(2):
    print(line[:200], "...")
    print("---")

ID,Name,LongName,photoUrl,playerUrl,Nationality,Age,↓OVA,POT,Club,Contract,Positions,Height,Weight,Preferred Foot,BOV,Best Position,Joined,Loan Date End,Value,Wage,Release Clause,Attacking,Crossing,Fi ...
---
158023,L. Messi,Lionel Messi,https://cdn.sofifa.com/players/158/023/21_60.png,http://sofifa.com/player/158023/lionel-messi/210006/,Argentina,33,93,93," ...
---


In [7]:
def parse_csv_line(line):
    """Parse a single CSV line into a list of field strings."""
    reader = csv.reader([line])
    return next(reader, [])

# Extract the header so we know how many columns each record should have.
header = rawRDD.first()
header_fields = parse_csv_line(header)
expected_n_cols = len(header_fields)

print(f"Expected number of columns: {expected_n_cols}")
print(f"First 5 column names: {header_fields[:5]}")

# Build a parsed RDD by removing the header and applying the parser to every line.
parsedRDD = rawRDD.filter(lambda l: l != header) \
                  .map(parse_csv_line)

print(f"\nParsed records: {parsedRDD.count()}")

Expected number of columns: 77
First 5 column names: ['ID', 'Name', 'LongName', 'photoUrl', 'playerUrl']

Parsed records: 93947


In [8]:
# Count how many parsed records have each possible number of fields.
field_count_distribution = parsedRDD.map(lambda r: (len(r), 1)) \
                                     .reduceByKey(lambda a, b: a + b) \
                                     .sortBy(lambda x: -x[1]) \
                                     .collect()

print(f"Records grouped by their field count:")
print(f"{'fields':>8}  {'records':>10}")
print("-" * 22)
for n_fields, count in field_count_distribution:
    print(f"{n_fields:>8}  {count:>10}")

# How many survive a strict "exactly the right number of columns" filter?
complete = parsedRDD.filter(lambda r: len(r) == expected_n_cols).count()
print(f"\nRecords with exactly {expected_n_cols} fields: {complete}")
print(f"That is {100 * complete / parsedRDD.count():.1f}% of all parsed records.")

Records grouped by their field count:
  fields     records
----------------------
       0       56226
      10       18742
      68       18742
      77         237

Records with exactly 77 fields: 237
That is 0.3% of all parsed records.
